In [38]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.nn import functional as F

In [39]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [40]:
print("Length (char):", len(text))

Length (char): 1115394


In [41]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [42]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [43]:
print(vocab_size)

65


In [44]:
str_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_str = { i:ch for i, ch in enumerate(chars) }

In [45]:
encode = lambda s: [str_to_int[c] for c in s] #take a str and output a list of int

In [46]:
decode = lambda l: ''.join([int_to_str[i] for i in l]) #vice versa

In [47]:
print(encode("Meoww"))

[25, 43, 53, 61, 61]


In [48]:
print(decode([25, 43, 53, 61, 61]))

Meoww


ENCODING ENTIRE TEXT DATASET and then store it into torch.Tensor

In [49]:
data = torch.tensor(encode(text), dtype =torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [50]:
print(data[:100])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [51]:
train, val = train_test_split(data, train_size=0.9, random_state=1)

In [52]:
print(train.shape)
print(val.shape)

torch.Size([1003854])
torch.Size([111540])


In [53]:
block_size = 8
train[:block_size+1] # efficiency + for making TF see context from as little as 1 to block size

#tf will never see more than block_size input and op is truncated. context for any thing between 1 to blocksize can be inferenced

tensor([40, 61, 52,  6, 46, 30, 28, 57, 42])

we will have mini batches of multiple chunks of text stacked up on a single tensor. FOR efficiency and parallelization. KEEP GPU BUSY.

In [54]:
torch.manual_seed(3407)
batch_size = 4 #indep sequence processing in parallel
block_size = 8 #maximum context length for prediction

def get_batch (split):
    data = train if split == 'train' else val
    ix = torch.randint( len(data) - block_size, (batch_size,))
    x = torch.stack(
        [data[i: i+block_size] for i in ix]
    )
    y = torch.stack(
            [data[i+1: i+block_size+1] for i in ix]
    )
    return x, y

In [55]:
xb, yb = get_batch('train')

In [56]:
print(xb.shape)
print(xb)

torch.Size([4, 8])
tensor([[53,  1, 56,  1, 57, 20, 52, 56],
        [13, 63, 56, 43, 51, 39, 43, 31],
        [ 1, 43, 47, 59, 56, 45, 53,  1],
        [ 1, 47, 51, 46, 18, 58, 46, 57]])


In [57]:
print(yb.shape)
print(yb)

torch.Size([4, 8])
tensor([[ 1, 56,  1, 57, 20, 52, 56,  8],
        [63, 56, 43, 51, 39, 43, 31,  0],
        [43, 47, 59, 56, 45, 53,  1, 58],
        [47, 51, 46, 18, 58, 46, 57, 26]])


In [58]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        #idx and targets are both (B, T) tensor
        logits = self.token_embedding_table(idx)  #(B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [59]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.6744, grad_fn=<NllLossBackward0>)


In [60]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


bUfoM$ tvHkYnGSz:NEl&m&XqXY!req?ZedOpjd ,&cJGGCsRjJlWxpji' fV&tHcKRj
UeiK&t-R:,V,&v&;YvaKOzWIBpMyixS


In [61]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3) #3e-4 for most networks but for small this works or even higher

In [62]:
batch_size = 32
for steps in range(10000):
    #sample a batch of data
    xb, yb = get_batch('train')

    #eval the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) #setting grads in previous step to zero
    loss.backward() #get grad for all the param
    optimizer.step() #use grad to update param

print(loss.item())

3.4404542446136475


In [65]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


-itoe :dCewI'y t,lehgTSdhoy  .d t l !UmtGdn sa  am,i EspchoelMTeNBo udrisky,Hs  
leme:onj-ut yr cDo'drdyTeIdumlwu eyvA3VelkRdrlkb tel
Vlht sz'ssfivbNakteh sohsoehhtl  ealno Iot srMl onl  ieeoylnhioeesIl'io  heAsistfni!U.e n
nIra:o hses mwanabnuyIhE'eglh
hio ot
onsW Wsst  's.c.en,ahh. tgaltlagyLthol

